In [ ]:
torch.cuda.empty_cache()
#gc.collect()


In [ ]:
import torch

In [ ]:
train_df = pd.read_excel("/content/cases_train2 (5).xlsx")
test_df = pd.read_excel("/content/Test1_2 (3).xlsx")


In [ ]:
import pandas as pd
import torch
import json
import re
import os
from datasets import Dataset
from transformers import TrainingArguments
from tqdm import tqdm

In [ ]:

!pip install llama-cpp-python -q
from llama_cpp import Llama
import torch
torch.cuda.empty_cache()
print("🔄 Загрузка GigaChat 3.1 Lightning из репозитория bartowski...")

model = Llama.from_pretrained(
    repo_id="bartowski/ai-sage_GigaChat3-10B-A1.8B-GGUF",
    filename="ai-sage_GigaChat3-10B-A1.8B-Q4_K_M.gguf",
    n_ctx=4096,
    n_gpu_layers=-1,
    verbose=False,
)


response = model.create_chat_completion(
    messages=[
        {"role": "system", "content": "Ты полезный ассистент."},
        {"role": "user", "content": "Напиши коротко о GigaChat."}
    ],
    max_tokens=256,
    temperature=0.7,
)

print(response['choices'][0]['message']['content'])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
🔄 Загрузка GigaChat 3.1 Lightning из репозитория bartowski...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./ai-sage_GigaChat3-10B-A1.8B-Q4_K_M.ggu(…): reconstructing file:   0%|          |  0.00B / 6.66GB            

./ai-sage_GigaChat3-10B-A1.8B-Q4_K_M.ggu(…): downloading bytes:           |  0.00B            

llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 1.0, mscale_all_dim = 1.0)


GigaChat — это языковая модель, разработанная Сбером. Она способна генерировать текст, отвечать на вопросы, переводить и выполнять другие задачи, связанные с обработкой естественного языка.


In [ ]:
def generate_critique(project_text, scores, model, max_new_tokens=1024):
    if pd.isna(project_text) or project_text == "":
        return "❌ Нет текста проекта"
    example_critique = """
ПРИМЕР КАЧЕСТВЕННОЙ КРИТИКИ:

1. АНАЛИЗ ЦА (оценка 1/5): Целевая аудитория не сегментирована и описана обобщенно как «таксопарки и водители». Отсутствует анализ конкретных сегментов (самозанятые водители, микропарки на 2-5 машин, крупные таксопарки), их проблем, барьеров и потребностей. Не представлена карта клиентского пути (CJM). Рекомендуется провести сегментацию ЦА и глубинные интервью с представителями каждого сегмента для выявления реальных болевых точек.

2. ПРОРАБОТКА РЕШЕНИЯ (оценка 1/5): Предложенное решение носит шаблонный характер («кредиты на авто и топливные карты») и не имеет уникальной концепции. Отсутствует MVP-стратегия, дорожная карта развития продукта, описание ключевых функций и этапов внедрения. Партнеры упомянуты без конкретных условий сотрудничества. Необходимо разработать комплексное решение с уникальным УТП, включая кредит с отсрочкой платежа, топливные карты с кэшбэком и страховку с покрытием рисков.

3. ФИНАНСОВАЯ МОДЕЛЬ (оценка 1/5): Финансовые показатели (3 млн рублей, 100 клиентов) не обоснованы и не подтверждены расчетами. Отсутствуют ключевые метрики: CAC (стоимость привлечения клиента), LTV (пожизненная ценность клиента), NPV (чистая приведенная стоимость), точка безубыточности, маржинальность. Нет сценарного анализа (оптимистичный, пессимистичный, реалистичный). Требуется детализированная финансовая модель с обоснованием всех показателей и прогнозом на 3-5 лет.

4. АНАЛИЗ РИСКОВ (оценка 1/5): Риски полностью отсутствуют, что нереалистично для любого бизнес-проекта. Нет количественной оценки вероятности и влияния рисков, отсутствуют планы митигации. Рекомендуется провести SWOT-анализ, описать основные риски (невозврат кредитов, аварийность, ценовая конкуренция, изменение законодательства) и разработать конкретные стратегии их снижения с указанием ответственных лиц.

5. ДОКАЗАТЕЛЬСТВА (оценка 2/5): Единственным доказательством является знакомый таксист как источник информации, что не является валидным подтверждением спроса. Упоминается статья без ссылки и конкретных данных. Не представлены результаты пилотных запусков, прототипов или интервью с потенциальными клиентами. Необходимо провести исследование рынка с использованием данных Росстата, интервью с 10+ владельцами таксопарков, собрать реальные метрики и documented кейсы.

Итоговая оценка: 1.2/5
Общий вывод: Проект находится на начальной стадии концепции и требует фундаментальной проработки по всем критериям. Основные направления доработки: сегментация ЦА, создание уникального решения с четким УТП, построение детальной финансовой модели с обоснованными расчетами, комплексный анализ рисков и сбор доказательной базы через пилотные запуски и глубинные интервью. Для успешной реализации рекомендуется привлечь экспертов по каждому из направлений.
"""

    system_prompt = f"""Ты — эксперт по оценке бизнес-проектов.

Вот оценки текущего проекта по 5 критериям (каждый от 1 до 5):
1. Анализ ЦА: {scores['ЦА']}/5
2. Проработка решения: {scores['Проработка']}/5
3. Финансовая модель: {scores['Финансы']}/5
4. Анализ рисков: {scores['Риски']}/5
5. Доказательства: {scores['Доказательства']}/5

{example_critique}

Напиши РАЗВЕРНУТУЮ КРИТИКУ для КОНКРЕТНОГО проекта, используя оценки выше.

КРИТИЧЕСКИ ВАЖНО:
- Пример выше показывает СТРУКТУРУ и СТИЛЬ, но НЕ КОПИРУЙ текст примера!
- Твоя критика должна быть УНИКАЛЬНОЙ для этого конкретного проекта
- Обязательно ссылайся на КОНКРЕТНЫЕ детали из текста проекта
- Объясни, ПОЧЕМУ проект получил каждую из указанных оценок
- Предложи КОНКРЕТНЫЕ улучшения, основанные на слабых местах проекта
- Если оценка высокая (4-5), объясни, что сделано хорошо и как это можно усилить
- Если оценка низкая (1-2), детально опиши проблемы и пути их решения
- Формат должен ТОЧНО соответствовать структуре примера
- Не используй смайлики, слэнг, сохраняй нейтральный деловой стиль

Формат ответа (строго соблюдай структуру):
1. АНАЛИЗ ЦА (оценка X/5): [уникальный комментарий по конкретному проекту]
2. ПРОРАБОТКА РЕШЕНИЯ (оценка X/5): [уникальный комментарий по конкретному проекту]
3. ФИНАНСОВАЯ МОДЕЛЬ (оценка X/5): [уникальный комментарий по конкретному проекту]
4. АНАЛИЗ РИСКОВ (оценка X/5): [уникальный комментарий по конкретному проекту]
5. ДОКАЗАТЕЛЬСТВА (оценка X/5): [уникальный комментарий по конкретному проекту]

Итоговая оценка: {sum(scores.values()) / 5:.1f}/5
Общий вывод: [уникальный вывод по конкретному проекту]

Помни: ты оцениваешь КОНКРЕТНЫЙ проект из текста ниже.
Не копируй пример, а создай уникальную критику на основе оценок и содержания проекта.
Твой ответ должен начинаться с "1. АНАЛИЗ ЦА" и строго следовать формату."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Вот текст проекта для оценки:\n\n{project_text[:3000]}"}
    ]

    try:
        response = model.create_chat_completion(
            messages=messages,
            max_tokens=max_new_tokens,
            temperature=0.8,
            top_p=0.95,
            repeat_penalty=1.1,
        )
        return response['choices'][0]['message']['content'].strip()
    except Exception as e:
        return f"❌ Ошибка: {str(e)}"

In [ ]:
import pandas as pd

In [ ]:
train_df = pd.read_excel("/content/cases_train2 (5).xlsx")
test_df = pd.read_excel("/content/Test1_2 (3).xlsx")


In [ ]:
test_df["Критика_модели"] = None


In [ ]:
for idx in tqdm(test_df.index, desc="Обработка кейсов"):
    project_text = test_df.loc[idx, "Решение кейса"]
    scores = {
        'ЦА': int(test_df.loc[idx, 'ЦА']) if 'ЦА' in test_df.columns else 3,
        'Проработка': int(test_df.loc[idx, 'Проработка решения']) if 'Проработка решения' in test_df.columns else 3,
        'Финансы': int(test_df.loc[idx, 'Финансовая модель и метрики']) if 'Финансовая модель и метрики' in test_df.columns else 3,
        'Риски': int(test_df.loc[idx, 'Анализ рисков']) if 'Анализ рисков' in test_df.columns else 3,
        'Доказательства': int(test_df.loc[idx, 'Доказательства']) if 'Доказательства' in test_df.columns else 3
    }

    critique = generate_critique(project_text, scores, model)
    test_df.loc[idx, "Критика_модели"] = critique
    if (idx + 1) % 5 == 0:
        test_df.to_excel("critique_gigachat_partial.xlsx", index=False)
        print(f"\n💾 Сохранено {idx + 1} кейсов")


Обработка кейсов:   4%|▍         | 5/130 [1:03:37<26:06:09, 751.75s/it]


💾 Сохранено 5 кейсов


Обработка кейсов:   8%|▊         | 10/130 [2:04:18<25:30:26, 765.22s/it]


💾 Сохранено 10 кейсов


Обработка кейсов:   9%|▉         | 12/130 [2:25:02<22:34:27, 688.71s/it]

In [ ]:
output_file = "test_with_critique_gigachat.xlsx"
test_df.to_excel(output_file, index=False)